# Bandwidth search on all DR options

Find optimal bandwidth for each class using a different DR options as the reduction input.

In [ ]:
import pathlib
from glob import glob

import geopandas as gpd
import joblib
import numpy as np
import pandas as pd

from gwlearn.ensemble import GWRandomForestClassifier
from gwlearn.linear_model import GWLogisticRegression
from gwlearn.search import BandwidthSearch

In [ ]:
census = gpd.read_parquet(
    "/data/uscuni-restricted/04_spatial_census/_merged_census_2021_relative_scaled.parquet"
)

In [ ]:
selection = [
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední vč. vyučení bez maturity - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední s maturitou vč. nástavbového a pomaturitního - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání:  vysokoškolské - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: nezjištěno - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem",
    "Zaměstnaní - Pracovníci ve službách a prodeji",
    "Zaměstnaní - Řemeslníci a opraváři",
    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: zaměstnanci - celkem",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý",
    "Počet osob v bytech celkem  s právním důvodem užívání: družstevní",
    "Počet obyvatel na byt",
    "Počet osob v domech celkem s vlastnictvím:  fyzická osoba",
    "Obyvatelstvo - věk: 0 - 6  - celkem",
    "Obyvatelstvo - věk: 7 - 14  - celkem",
    "Obyvatelstvo - věk: 15 - 24  - celkem",
    "Obyvatelstvo - věk: 45 - 54  - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby na rodičovské dovolené - celkem",
    "Obyvatelstvo - státní občanství: Slovenská republika - celkem",
    "Obyvatelstvo - státní občanství: země EU mimo ČR - celkem",
    "Obyvatelstvo - státní občanství: nezjištěno - celkem",
    "Obyvatelstvo - náboženská víra: bez náboženské víry - celkem",
    "Obyvatelstvo - náboženská víra: neuvedeno - celkem",
    "Obyvatelstvo - s trvalým pobytem - celkem",
    "Obyvatelstvo - rodinný stav: ženatí, vdané - celkem",
    "Obyvatelstvo - rodinný stav: rozvedení - celkem",
    "Obyvatelstvo - rodinný stav: ovdovělí - celkem",
    "geometry",
]

In [ ]:
# Load data
clusters = pd.read_csv(
    "/data/uscuni-restricted/04_spatial_census/cluster_assignment_v10.csv",
    dtype={"kod_nadzsj_d": str},
)
cluster_mapping = pd.read_parquet(
    "/data/uscuni-ulce/processed_data/clusters/cluster_mapping_v10.pq"
)
data = census.merge(clusters, left_on="nadzsjd", right_on="kod_nadzsj_d")
variables = data.columns.drop(["geometry", "kod_nadzsj_d", "final_without_noise"])

mapped = data["final_without_noise"].map(cluster_mapping[3])

In [ ]:
# adaptive bandwidth search
for label in [1, 3, 7, 8, 4, 5]:
    y = mapped == label

    print(f"Label: {label}")
    search = BandwidthSearch(
        GWRandomForestClassifier,
        fixed=False,
        n_jobs=-1,
        search_method="interval",
        min_bandwidth=50,
        max_bandwidth=5000,
        interval=450,
        criterion="aicc",
        metrics=["aic", "bic", "aicc", "log_loss", "prediction_rate"],
        verbose=True,
        batch_size=1000,
        min_proportion=0.08,
        class_weight="balanced",
        undersample=True,
        random_state=42,
    )
    search.fit(
        data[variables],
        y,
        geometry=data.representative_point(),
    )
    search.metrics_.to_csv(
        f"/data/uscuni-restricted/06_bandwidths/{label}_vs_fa_adaptive_3.csv"
    )

In [ ]:
# adaptive bandwidth search
for label in [2, 6]:
    y = mapped == label

    print(f"Label: {label}")
    search = BandwidthSearch(
        GWRandomForestClassifier,
        fixed=False,
        n_jobs=-1,
        search_method="interval",
        min_bandwidth=50,
        max_bandwidth=1300,
        interval=150,
        criterion="aicc",
        metrics=["aic", "bic", "aicc", "log_loss", "prediction_rate"],
        verbose=True,
        batch_size=1000,
        min_proportion=0.08,
        class_weight="balanced",
        undersample=True,
        random_state=42,
    )
    search.fit(
        data[variables],
        y,
        geometry=data.representative_point(),
    )
    search.metrics_.to_csv(
        f"/data/uscuni-restricted/06_bandwidths/{label}_vs_fa_adaptive_3.csv"
    )